# Guía de Laboratorio N.° 1 — Fundamentos y Entornos de Desarrollo en Python
### Tecnologías Emergentes (ISO46B) · Semana 1 · Facultad de Ingeniería de Sistemas — UNCP

**Versión: guía de trabajo del estudiante.**

Duración: 4 horas. Ejecute las celdas en orden. Los comandos usan rutas directas al entorno virtual (`.venv/bin/...`) en lugar de `activate`, porque cada celda de Jupyter que empieza con `!` se ejecuta en un subproceso nuevo y no conserva la activación entre celdas. **En Windows, fuera del notebook, use `.venv\\Scripts\\...`.**

## Requisitos previos
- Python 3.10+
- Git instalado
- Cuenta en GitHub
- Kernel de Jupyter con acceso a una terminal Unix-like (en Windows local, ejecute los pasos de Git/venv equivalentes desde PowerShell si el notebook falla).

## Parte 1 — Entornos virtuales y gestión de dependencias (60 min) · Nivel: Básico

### Paso 1.1 — Verificación de requisitos

In [ ]:
!python --version
!pip --version
!git --version

### Paso 1.2 — Creación del entorno virtual

In [ ]:
!python -m venv .venv

### Paso 1.3 — Instalación de paquetes y registro de dependencias

In [ ]:
!.venv/bin/pip install requests
!.venv/bin/pip list

In [ ]:
!.venv/bin/pip freeze > requirements.txt
!cat requirements.txt

### Paso 1.4 — Mini-reto: primer script funcional

In [ ]:
%%writefile main.py
import requests

respuesta = requests.get("https://official-joke-api.appspot.com/random_joke")
datos = respuesta.json()
print(datos["setup"])
print(datos["punchline"])


In [ ]:
!.venv/bin/python main.py

**Checkpoint de Parte 1:** el script debe ejecutarse sin errores antes de continuar.

## Parte 2 — Control de versiones con Git aplicado a Python (90 min) · Nivel: Intermedio

### Paso 2.1 — Configuración inicial de Git

In [ ]:
!git config --global user.name "Su Nombre"
!git config --global user.email "su_correo@uncp.edu.pe"
!git init

### Paso 2.2 — Creación de .gitignore y primer commit

In [ ]:
%%writefile .gitignore
.venv/
__pycache__/
*.pyc
.env
*.egg-info/
dist/
build/
.pytest_cache/


In [ ]:
!git add .gitignore requirements.txt main.py
!git commit -m "chore: configura entorno virtual y primer script funcional"

### Paso 2.3 — Trabajo con ramas

In [ ]:
!git checkout -b feature/manejo-errores

In [ ]:
%%writefile main.py
import requests

try:
    respuesta = requests.get(
        "https://official-joke-api.appspot.com/random_joke", timeout=5
    )
    respuesta.raise_for_status()
    datos = respuesta.json()
    print(datos["setup"])
    print(datos["punchline"])
except requests.RequestException as error:
    print(f"No se pudo obtener el chiste: {error}")


In [ ]:
!git add main.py
!git commit -m "feat: agrega manejo de errores en la consulta a la API"

### Paso 2.4 — Repositorio remoto y Pull Request en GitHub
Reemplace `SU-USUARIO` por su usuario real de GitHub antes de ejecutar. Los pasos de abrir y fusionar el Pull Request se hacen en la interfaz web de GitHub, no en el notebook.

In [ ]:
# Reemplace SU-USUARIO por su usuario de GitHub
!git remote add origin https://github.com/SU-USUARIO/tecemer-lab1.git
!git push -u origin main
!git push -u origin feature/manejo-errores
# ... abra un Pull Request de feature/manejo-errores hacia main en github.com y fusiónelo ...
!git checkout main
!git pull origin main

**Checkpoint de Parte 2:** revise en GitHub que su Pull Request fue fusionado correctamente.

## Parte 3 — Estructura de proyecto, PEP 8 y consolidación (90 min) · Nivel: Avanzado

### Paso 3.1 — Migración a src layout

In [ ]:
import os

os.makedirs("src/tecemer_lab1", exist_ok=True)
os.rename("main.py", "src/tecemer_lab1/app.py")
open("src/tecemer_lab1/__init__.py", "w").close()
print("Reestructuracion completa.")


In [ ]:
from pathlib import Path

def imprimir_arbol(base=".", prefijo="", excluir=(".venv", ".git", "__pycache__")):
    base = Path(base)
    entradas = sorted([p for p in base.iterdir() if p.name not in excluir])
    for i, p in enumerate(entradas):
        conector = "\u2514\u2500\u2500 " if i == len(entradas) - 1 else "\u251c\u2500\u2500 "
        print(prefijo + conector + p.name + ("/" if p.is_dir() else ""))
        if p.is_dir():
            extension = "    " if i == len(entradas) - 1 else "\u2502   "
            imprimir_arbol(p, prefijo + extension, excluir)

imprimir_arbol(".")


### Paso 3.2 — pyproject.toml e instalación editable

In [ ]:
%%writefile pyproject.toml
[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "tecemer-lab1"
version = "0.1.0"
requires-python = ">=3.10"
dependencies = [
    "requests>=2.28",
]


In [ ]:
!.venv/bin/pip install -e .

### Paso 3.3 — Estilo de código con PEP 8

In [ ]:
!.venv/bin/pip install black pycodestyle

In [ ]:
!.venv/bin/pycodestyle src/

In [ ]:
!.venv/bin/black src/

In [ ]:
!git add pyproject.toml src/
!git commit -m "style: aplica formateo PEP 8 con Black y agrega pyproject.toml"

### Paso 3.4 — Documentación con README.md

In [ ]:
%%writefile README.md
# tecemer-lab1

Proyecto de práctica de la Semana 1 del curso Tecnologías Emergentes (ISO46B) — UNCP.
Consume una API pública de chistes como ejercicio de configuración de entorno.

## Instalación

```bash
python -m venv .venv
source .venv/bin/activate   # Windows: .venv\Scripts\activate
pip install -e .
```

## Uso

```bash
python -m tecemer_lab1.app
```

## Estructura del repositorio

```
tecemer-lab1/
├── src/tecemer_lab1/   # código fuente
├── pyproject.toml      # metadatos y dependencias
├── README.md
└── .gitignore
```

## Autor

Curso: Tecnologías Emergentes (ISO46B) — Facultad de Ingeniería de Sistemas, UNCP.


In [ ]:
!git add README.md
!git commit -m "docs: agrega README con instrucciones de instalación y uso"

### Paso 3.5 — Consolidación final y entrega

In [ ]:
!git push origin main

Copie el enlace de su repositorio de GitHub y súbalo al aula virtual como evidencia de entrega.

## Entregable de la sesión
Repositorio en GitHub con: estructura en src layout, entorno virtual documentado (no subido), pyproject.toml funcional con instalación editable, código formateado según PEP 8, README.md completo, y un historial de al menos 5 commits en formato Conventional Commits, incluyendo al menos una rama de feature fusionada mediante Pull Request.

## Checklist final rápido
- [ ] El entorno virtual (`.venv`) existe y NO fue subido al repositorio.
- [ ] `.gitignore` excluye `.venv/`, `__pycache__/` y archivos temporales.
- [ ] Existe `pyproject.toml` y `pip install -e .` se ejecuta sin errores.
- [ ] El código en `src/` fue formateado con Black y pasa `pycodestyle` sin observaciones críticas.
- [ ] El `README.md` incluye instalación, uso, estructura y autor.
- [ ] El historial de commits sigue Conventional Commits (`feat:`, `fix:`, `chore:`, `docs:`, `style:`).
- [ ] Existe al menos un Pull Request fusionado en el repositorio remoto.
